# BOLT by example

This notebook is an end-to-end tour of the **Books of Life Toolkit (BOLT)**: it
transforms *complex log data* (registry-style records spread across many rows and
tables) into a single plain-text **book of life** per person, ready for analysis
with large language models.

It walks through:

1. Connecting to and inspecting the database
2. Reading a recipe (the *what / who / how*)
3. Writing a single book of life
4. Adding **social context** (books within books)
5. Defining a recipe inline in Python
6. Generating books at scale and saving to JSONL
7. Reading the saved dataset back

**Prerequisites.** Run from the repository root with the package installed
(`pip install -e .`). No data is required up front: the first code cell builds
`dbs/db.duckdb` from synthetic data automatically if it is missing.

**Tip:** use *Run All*, or at minimum run the **setup cell** (imports + database)
and **section 1** (connect to the database) before the later sections — they define
`conn`, `person`, and `rinpersoons` used throughout.

In [1]:
import os
import sys
import json
import subprocess

import duckdb

# Run this notebook from the repository root so relative paths
# (dbs/, recipes/, synth/) resolve correctly.
if not os.path.exists("main.py"):
    raise RuntimeError(
        "Please run this notebook from the BooksOfLifeToolkit repository root."
    )

from serialization.Recipe import Recipe
from serialization.registry import INSTANTIATORS
from serialization.BookofLifeGenerator import BookofLifeGenerator
from serialization.BookofLifeGeneratorBatch import BookofLifeGeneratorBatch
from utils.utils import get_unique_rinpersoons, basic_gen, save_to_jsonl_shard

DB_PATH = "dbs/db.duckdb"

# BOLT ships no data. dbs/db.duckdb is built locally from synthetic data.
# If it does not exist yet, build it now: synthetic data -> CSVs -> DuckDB.
if not os.path.exists(DB_PATH):
    print("No database found - building one from synthetic data...")
    subprocess.run([sys.executable, "synth/main.py"], check=True)
    subprocess.run(
        [
            sys.executable, "serialization/make_db.py",
            "--data_dir", "synth/data",
            "--yaml_file", "recipes/make_db",
            "--db_name", "db",
        ],
        check=True,
    )
    print("Done.")
else:
    print(f"Using existing database at {DB_PATH}.")

Using existing database at dbs/db.duckdb.


## 1. Connect to and inspect the database

BOLT reads from a DuckDB database whose tables are "complex log data" sources
(here: demographics and household spells). Notice how one person's life is
spread across many rows in `household_bus` — exactly the structure BOLT turns
into a single readable narrative.

In [2]:
conn = duckdb.connect(DB_PATH, read_only=True)

# Pick one example person up front so later cells can be run independently
rinpersoons = get_unique_rinpersoons(DB_PATH)
person = rinpersoons[0]

# What tables (data sources) are in the database?
tables = [t[0] for t in conn.execute("SHOW TABLES").fetchall()]
print("Tables:", tables)
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n:,} rows")
print(f"\nExample person for demos below: {person} ({len(rinpersoons):,} people total)")

# Peek at a few raw rows of the household log - the kind of "complex log data"
# BOLT turns into text. A single person's life is spread across many rows.
print("\nSample household_bus rows (raw registry-style records):")
conn.sql(
    "SELECT rinpersoon, HUISHOUDNR, TYPHH, PLHH, DATUMAANVANGHH, DATUMEINDEHH "
    "FROM household_bus LIMIT 5"
).show()

Number of rows in persoon_tab: 12229
Tables: ['household_bus', 'persoon_tab']
  household_bus: 112,037 rows
  persoon_tab: 12,229 rows

Example person for demos below: 003fbfa0 (12,229 people total)

Sample household_bus rows (raw registry-style records):
┌────────────┬────────────┬─────────┬─────────┬────────────────┬──────────────┐
│ rinpersoon │ HUISHOUDNR │  TYPHH  │  PLHH   │ DATUMAANVANGHH │ DATUMEINDEHH │
│  varchar   │  varchar   │ varchar │ varchar │    varchar     │   varchar    │
├────────────┼────────────┼─────────┼─────────┼────────────────┼──────────────┤
│ fe39c796   │ 9beffcf1   │ 3       │ 3       │ 19900101       │ 19910101     │
│ 1f957d0a   │ 9beffcf1   │ 3       │ 3       │ 19900101       │ 19910101     │
│ 25945fe1   │ 21c9d5ea   │ 5       │ 4       │ 19900101       │ 19910101     │
│ 64d6351a   │ 21c9d5ea   │ 5       │ 4       │ 19900101       │ 19910101     │
│ f66bd07d   │ 21c9d5ea   │ 5       │ 1       │ 19900101       │ 19910101     │
└────────────┴──────────

## 2. The recipe: *what*, *who*, *how*

A **recipe** is the configuration that drives a book. It specifies the **what**
(which data sources and features to include), the **who** (social context), and
the **how** (ordering and formatting). Let's look at the template recipe.

In [3]:
# A recipe declares the WHAT (information sources + features), the WHO
# (social context), and the HOW (ordering + formatting).
with open("recipes/template.yaml") as f:
    print(f.read())

recipe = Recipe("recipes/template.yaml")
print("main_key:         ", recipe.main_key)
print("datasets (what):  ", recipe.dataset_names)
print("sorting (how):    ", recipe.sorting_keys)
print("generator (how):  ", recipe.paragraph_generator)
print("registered sources:", sorted(INSTANTIATORS))

main_key: rinpersoon
datasets:
  # The "what": demographic information about the focal person plus their full
  # household history. Mirrors the "Book 3" style from the paper.
  - name: persoon_tab
    features:
      - GBAGEBOORTEJAAR
      - GBAGESLACHT
      - GBAGEBOORTELAND
      - GBAGEBOORTEJAARVADER
      - GBAGESLACHTVADER
      - GBAGEBOORTEJAARMOEDER
      - GBAGESLACHTMOEDER
  - name: household_bus
    features:
      - HUISHOUDNR
      - TYPHH
      - DATUMAANVANGHH
      - DATUMEINDEHH
      - PLHH
      - REFPERSOONHH
      - AANTALPERSHH
      - AANTALKINDHH
formatting:
  # The "how": order paragraphs chronologically and render key/value pairs.
  sorting_keys:
      - year
  paragraph_generator: get_paragraph_string_tabular

main_key:          rinpersoon
datasets (what):   ['persoon_tab', 'household_bus']
sorting (how):     ['year']
generator (how):   get_paragraph_string_tabular
registered sources: ['household_bus', 'persoon_tab']


## 3. Write a single Book of Life

`BookofLifeGenerator` reads the recipe, pulls the relevant rows for one person
from the database, turns each into a paragraph, orders them, and renders the
final plain-text book.

In [4]:
# Write one person's book from the template recipe (uses `person` from section 1).
book = BookofLifeGenerator(person, "recipes/template.yaml", duck_db_conn=conn).generate_book()
print(f"Book of Life for {person}\n" + "=" * 60)
print(book)

Book of Life for 003fbfa0


Country of Birth: NL
Gender: 2
Year of Birth: 1987
Mother's Gender: 2
Mother's Year of Birth: 1962
Father's Gender: 1
Father's Year of Birth: 1953

Household ID: a44d8ee0
Type of Household: married couple with children
Date Household Began: Jan 01 1990
Date Household Ended: Jan 01 1991
Number of People in Household: 5
Role in Household: partner
Is Reference Person: yes
Number of Children in Household: 3

Household ID: a44d8ee0
Type of Household: married couple with children
Date Household Began: Jan 01 1991
Date Household Ended: Jan 01 1992
Number of People in Household: 6
Role in Household: partner
Is Reference Person: yes
Number of Children in Household: 4

Household ID: a44d8ee0
Type of Household: married couple with children
Date Household Began: Jan 01 1992
Date Household Ended: Jan 01 1993
Number of People in Household: 6
Role in Household: partner
Is Reference Person: yes
Number of Children in Household: 4

Household ID: a44d8ee0
Type of Household: ma

## 4. Social context: books within books

A key strength of BOLT is the **who**: capturing the linked lives around a person.
The `social_context` recipe writes short nested books for the partners and children
in each household spell, embedded inside the focal person's book.

In [ ]:
# The "who": include other people (partners, children) as nested
# "books within books". Find someone who is a partner in a household with children.
row = conn.execute(
    """
    SELECT rinpersoon, HUISHOUDNR, DATUMAANVANGHH
    FROM household_bus
    WHERE PLHH IN ('3', '4')          -- partner in household
      AND AANTALKINDHH NOT IN ('0', 'nan')
    LIMIT 1
    """
).fetchone()
if row is None:
    raise RuntimeError(
        "No partner-with-children found in household_bus. "
        "Re-run the setup cell to rebuild synthetic data."
    )
focal, hh_id, spell_start = row
print(f"Focal person: {focal}  |  household {hh_id}  |  spell starts {spell_start}")

# The social_context recipe adds PARTNERS and CHILDREN sub-books under each
# household spell. Compare with recipes/social_context.yaml.
with open("recipes/social_context.yaml") as f:
    print("\n--- recipes/social_context.yaml ---")
    print(f.read())

social_book = BookofLifeGenerator(
    focal, "recipes/social_context.yaml", duck_db_conn=conn
).generate_book()

# The full book can be long (one block per household spell). Show the first
# spell that contains nested sub-books so the structure is easy to read.
blocks = [b.strip() for b in social_book.split("\n\n") if b.strip()]
first_spell_with_context = next(
    b for b in blocks if "[PARTNERS" in b or "[CHILDREN" in b
)
print(f"\nBook of Life (with social context) for {focal}")
print("=" * 60)
print("First household spell with nested sub-books:\n")
print(first_spell_with_context)
print("\n... (later spells omitted; run social_book in full to see the rest)")

Focal person: 25945fe1  |  household 21c9d5ea  |  spell starts 19900101

--- recipes/social_context.yaml ---
main_key: rinpersoon
datasets:
  # The "what": information about the focal person.
  - name: persoon_tab
    features:
      - GBAGEBOORTEJAAR
      - GBAGESLACHT
  - name: household_bus
    features:
      - HUISHOUDNR
      - TYPHH
      - DATUMAANVANGHH
      - DATUMEINDEHH
      - PLHH
    # The "who": for every household spell, write short "books within books"
    # for the partners and children present in that spell.
    social_context_features:
      PARTNERS:
        - name: persoon_tab
          features:
            - GBAGEBOORTEJAAR
            - GBAGESLACHT
      CHILDREN:
        - name: persoon_tab
          features:
            - GBAGEBOORTEJAAR
formatting:
  # The "how": order paragraphs chronologically and render key/value pairs.
  sorting_keys:
      - year
  paragraph_generator: get_paragraph_string_tabular


Book of Life (with social context) for 25945fe1
Fi

## 5. Define a recipe inline

Recipes are just dictionaries, so you can build them in Python without a YAML
file. This makes it easy to experiment with different *what / who / how* choices.

In [ ]:
# Recipes can also be built in Python as a dict - handy for experimentation.
# Here: a minimal book with only sex and year of birth.
inline_recipe = {
    "main_key": "rinpersoon",
    "datasets": [
        {"name": "persoon_tab", "features": ["GBAGESLACHT", "GBAGEBOORTEJAAR"]},
    ],
    "formatting": {
        "sorting_keys": ["year"],
        "paragraph_generator": "get_paragraph_string_tabular",
    },
}

mini_book = BookofLifeGenerator(person, inline_recipe, duck_db_conn=conn).generate_book()
print(f"Minimal book for {person}\n" + "=" * 60)
print(mini_book)

Minimal book for 003fbfa0


Gender: 2
Year of Birth: 1987


## 6. Generate at scale and save to JSONL

For real use you generate books for many people at once. `BookofLifeGeneratorBatch`
instantiates paragraphs for the whole population in one pass; the books are then
written out and saved as a sharded JSONL file (`output/books.jsonl`).

In [8]:
# Generate books at scale and save them as JSONL (one record per person).
batch_generator = BookofLifeGeneratorBatch(rinpersoons, "recipes/template.yaml", DB_PATH, conn)
batch_generator.write_books()

data_buffer = []
for rinpersoon, paragraphs in batch_generator.rin_dicts.items():
    rid, book_content = basic_gen(
        rinpersoon=rinpersoon,
        recipe_yaml_path="recipes/template.yaml",
        paragraphs=paragraphs,
        conn=conn,
    )
    data_buffer.append({"rinpersoon": rid, "book_content": book_content})

# Start from a clean file so re-running the notebook does not append duplicates.
out_file = os.path.join("output", "books.jsonl")
if os.path.exists(out_file):
    os.remove(out_file)
out_path = save_to_jsonl_shard(data_buffer, "output", shard_index=None)
print(f"Wrote {len(data_buffer):,} books to {out_path}")

char_lengths = [len(r["book_content"]) for r in data_buffer]
word_lengths = [len(r["book_content"].split()) for r in data_buffer]
print(
    f"Avg length: {sum(char_lengths) // len(char_lengths):,} chars / "
    f"{sum(word_lengths) // len(word_lengths):,} words per book"
)

Wrote 12,229 books to output/books.jsonl
Avg length: 2,436 chars / 395 words per book


## 7. Inspect the saved dataset

Finally, read the JSONL back and look at one record the way a downstream
consumer (e.g. an LLM fine-tuning or prompting pipeline) would.

In [9]:
# Read the saved dataset back and show one record as it would be consumed downstream.
with open(os.path.join("output", "books.jsonl")) as f:
    first = json.loads(f.readline())

print("Keys:", list(first.keys()))
print("rinpersoon:", first["rinpersoon"])
print("\nbook_content (first 600 chars):\n")
print(first["book_content"][:600])

Keys: ['rinpersoon', 'book_content']
rinpersoon: 003fbfa0

book_content (first 600 chars):



Country of Birth: NL
Gender: 2
Year of Birth: 1987
Mother's Gender: 2
Mother's Year of Birth: 1962
Father's Gender: 1
Father's Year of Birth: 1953

Household ID: a44d8ee0
Type of Household: married couple with children
Date Household Began: Jan 01 1990
Date Household Ended: Jan 01 1991
Number of People in Household: 5
Role in Household: partner
Is Reference Person: yes
Number of Children in Household: 3

Household ID: a44d8ee0
Type of Household: married couple with children
Date Household Began: Jan 01 1991
Date Household Ended: Jan 01 1992
Number of People in Household: 6
Role in Household:
